# 🤰 Onamiz — AI Model Training (v3 — O'zbekiston ma'lumotlari bilan)

**Ma'lumot manbalari:**
- WHO ANC Guidelines 2016
- ACOG Practice Bulletins 2023  
- FIGO Pre-eclampsia Guidelines 2022
- EPDS (Edinburgh Postnatal Depression Scale)
- **O'zbekiston SSV statistikasi 2023** (yangi!)
- **Andijan viloyati ona o'limi tadqiqoti 2025** (yangi!)
- **Markaziy Osiyo o'smirlar homiladorligi tadqiqoti** (yangi!)
- Kaggle: Maternal Health Risk + Fetal Health CTG

## O'zbekiston uchun maxsus ma'lumotlar (tadqiqot asosida):
| Ko'rsatkich | O'zbekiston statistikasi | Manba |
|---|---|---|
| Ona o'limi sababi #1 | Preeklampsia **24%** | Andijan tadqiqoti 2025 |
| Ona o'limi sababi #2 | Obstetrik qon ketish **24%** | Andijan tadqiqoti 2025 |
| Ona o'limi sababi #3 | Qo'shma kasalliklar **27-30%** | SSV 2023 |
| Anemiya tarqalishi | **30-74%** (o'smirda yuqori) | WHO Uzbekistan |
| O'smirlar STI xavfi | **21.5%** vs 6.6% kattalar | Qozog'iston tadqiqoti |
| Erta nikoh yoshlari | **66.3%** = 17 yosh | Demografiya.uz 2021 |
| Muddatidan oldin tug'ruq | **10.3%** o'smirlarda | Markaziy Osiyo tadqiqoti |


In [ ]:
!pip install -q pandas numpy scikit-learn xgboost lightgbm joblib matplotlib seaborn imbalanced-learn shap

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/onamiz_data', exist_ok=True)
os.makedirs('/content/drive/MyDrive/onamiz_models', exist_ok=True)
print('✅ Drive ulandi')

In [ ]:
import warnings, json, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import joblib
warnings.filterwarnings('ignore')
RANDOM_STATE = 42
plt.style.use('seaborn-v0_8-whitegrid')
print('✅ Kutubxonalar tayyor')

## 🌍 1-BLOK: Global dataset (WHO/ACOG/FIGO)

In [ ]:
np.random.seed(RANDOM_STATE)

def generate_global_dataset(n=3000):
    """WHO/ACOG/FIGO protokollariga asoslangan global dataset."""
    records = []
    for _ in range(n):
        trimester = np.random.choice(['T1','T2','T3'], p=[0.35, 0.35, 0.30])
        age = np.random.randint(16, 45)
        week = {'T1': np.random.randint(4,13), 'T2': np.random.randint(13,27), 'T3': np.random.randint(27,41)}[trimester]

        vaginal_bleeding    = np.random.choice([0,1,2], p=[0.80,0.15,0.05])
        one_sided_pain      = np.random.choice([0,1],   p=[0.93,0.07])
        nausea_severity     = np.random.choice([0,1,2,3,4], p=[0.20,0.30,0.30,0.15,0.05])
        urinary_burning     = np.random.choice([0,1],   p=[0.85,0.15])
        fever               = np.random.choice([0,1,2], p=[0.85,0.10,0.05])
        dizziness           = np.random.choice([0,1,2], p=[0.70,0.20,0.10])
        prev_miscarriage    = np.random.choice([0,1,2], p=[0.75,0.15,0.10])
        headache_severity   = np.random.choice([0,1,2], p=[0.70,0.20,0.10])
        visual_disturbance  = np.random.choice([0,1],   p=[0.95,0.05])
        edema_level         = np.random.choice([0,1,2,3], p=[0.55,0.25,0.15,0.05])
        fetal_movement      = np.random.choice([0,1,2], p=[0.80,0.15,0.05])
        epigastric_pain     = np.random.choice([0,1,2], p=[0.80,0.15,0.05])
        sudden_weight_gain  = np.random.choice([0,1],   p=[0.90,0.10])
        fluid_leaking       = np.random.choice([0,1],   p=[0.97,0.03])
        fetal_movement_t3   = np.random.choice([0,1,2], p=[0.78,0.17,0.05])
        contractions        = np.random.choice([0,1,2], p=[0.75,0.17,0.08])
        bleeding_with_pain  = np.random.choice([0,1],   p=[0.97,0.03])
        shortness_of_breath = np.random.choice([0,1,2], p=[0.75,0.20,0.05])
        systolic_bp         = np.random.normal(115, 15)
        diastolic_bp        = np.random.normal(75, 10)
        heart_rate          = np.random.normal(82, 12)
        # O'zbekiston xususiy (global uchun o'rtacha)
        anemia_level        = np.random.choice([0,1,2,3], p=[0.60,0.25,0.12,0.03])
        iron_supplement     = np.random.choice([0,1], p=[0.50,0.50])
        parity              = np.random.choice([0,1,2], p=[0.40,0.40,0.20])
        prenatal_visits     = np.random.choice([0,1,2], p=[0.10,0.40,0.50])
        nutrition_poor      = np.random.choice([0,1], p=[0.75,0.25])
        rural               = np.random.choice([0,1], p=[0.60,0.40])
        pph_history         = np.random.choice([0,1], p=[0.92,0.08])

        score = 0
        if vaginal_bleeding==2:    score+=10
        if one_sided_pain==1:      score+=10
        if visual_disturbance==1:  score+=10
        if fetal_movement==2:      score+=10
        if fluid_leaking==1:       score+=10
        if bleeding_with_pain==1:  score+=10
        if fetal_movement_t3==2:   score+=10
        if headache_severity==2:   score+=5
        if epigastric_pain==2:     score+=5
        if contractions==2:        score+=5
        if systolic_bp>=160:       score+=5
        if systolic_bp>=140:       score+=5
        if diastolic_bp>=90:       score+=5
        if fever==2:               score+=4
        if nausea_severity==4:     score+=4
        if shortness_of_breath==2: score+=4
        if vaginal_bleeding==1:    score+=3
        if nausea_severity==3:     score+=3
        if urinary_burning==1:     score+=3
        if dizziness==2:           score+=3
        if fetal_movement==1:      score+=3
        if fetal_movement_t3==1:   score+=3
        if edema_level==3:         score+=3
        if headache_severity==1:   score+=2
        if edema_level==2:         score+=2
        if sudden_weight_gain==1:  score+=2
        if prev_miscarriage==2:    score+=2
        if fever==1:               score+=2
        if contractions==1:        score+=2
        if epigastric_pain==1:     score+=2
        if shortness_of_breath==1: score+=2
        if anemia_level==2:        score+=2
        if anemia_level==3:        score+=3
        if age<18 or age>40:       score+=1
        if prev_miscarriage==1:    score+=1
        if pph_history==1:         score+=2
        if anemia_level==1:        score+=1
        if prenatal_visits==0:     score+=1
        if nutrition_poor==1:      score+=1

        risk = 'emergency' if score>=10 else 'high' if score>=6 else 'medium' if score>=3 else 'low'

        records.append({
            'trimester': trimester, 'age': age, 'gestational_week': week,
            'vaginal_bleeding': vaginal_bleeding, 'one_sided_pain': one_sided_pain,
            'nausea_severity': nausea_severity, 'urinary_burning': urinary_burning,
            'fever': fever, 'dizziness': dizziness, 'prev_miscarriage': prev_miscarriage,
            'headache_severity': headache_severity, 'visual_disturbance': visual_disturbance,
            'edema_level': edema_level, 'fetal_movement': fetal_movement,
            'epigastric_pain': epigastric_pain, 'sudden_weight_gain': sudden_weight_gain,
            'fluid_leaking': fluid_leaking, 'fetal_movement_t3': fetal_movement_t3,
            'contractions': contractions, 'bleeding_with_pain': bleeding_with_pain,
            'shortness_of_breath': shortness_of_breath,
            'systolic_bp': round(systolic_bp,1), 'diastolic_bp': round(diastolic_bp,1),
            'heart_rate': round(heart_rate,1),
            'anemia_level': anemia_level, 'iron_supplement': iron_supplement,
            'parity': parity, 'prenatal_visits': prenatal_visits,
            'nutrition_poor': nutrition_poor, 'rural': rural, 'pph_history': pph_history,
            'source': 'global', 'risk_level': risk
        })
    return pd.DataFrame(records)

df_global = generate_global_dataset(3000)
print('Global dataset:', df_global.shape)
print(df_global['risk_level'].value_counts())

## 🇺🇿 2-BLOK: O'zbekiston-xususiy dataset

### Ilmiy manbalar:
- **Andijan viloyati (2025):** Ona o'limi sabablari — preeklampsia 24%, qon ketish 24%, qo'shma kasalliklar 27-30%
- **WHO Uzbekistan:** Anemiya 30-74% (o'smirda 74.1%, kattalarda 67.9%)
- **Demografiya.uz (2021):** Erta nikoh — 66.3% = 17 yosh, 29.3% = 16 yosh
- **Markaziy Osiyo (2024):** O'smir onalar — STI 21.5%, muddatidan oldin tug'ruq 10.3%
- **SSV protokoli 2021:** Anemiyada temir qabul qilish majburiy

In [ ]:
def generate_uzbekistan_dataset(n=2000):
    """
    O'zbekiston-xususiy xavf omillariga asoslangan dataset.

    Asosiy farqlar (global vs O'zbekiston):
    1. Anemiya: 30-74% (global 15-20%)
    2. Erta nikoh/homiladorlik: 15-18 yosh ko'p uchraydi
    3. Ko'p homiladorlik (parity): qishloqlarda 3+ ko'p
    4. Temir qabul qilmaslik: ko'pchilik qabul qilmaydi
    5. Qishloq joylashuv: tibbiy xizmatga yetish qiyin
    6. STI xavfi: o'smirlarda 21.5%
    7. Obstetrik qon ketish: ona o'limining 24% sababi
    """
    records = []
    for _ in range(n):
        trimester = np.random.choice(['T1','T2','T3'], p=[0.35,0.35,0.30])

        # O'ZBEKISTON: yosh taqsimoti — ko'proq yosh onalar
        age = np.random.choice(
            list(range(15, 45)),
            p=[0.03,0.05,0.07,0.08,  # 15-18
               0.09,0.09,0.08,0.07,  # 19-22
               0.07,0.06,0.05,0.04,  # 23-26
               0.04,0.04,0.04,0.03,  # 27-30
               0.03,0.03,0.02,0.02,  # 31-34
               0.02,0.02,0.01,0.01,  # 35-38
               0.01,0.01,0.01,0.01,  # 39-42
               0.005,0.005]          # 43-44
        )
        week = {'T1': np.random.randint(4,13), 'T2': np.random.randint(13,27), 'T3': np.random.randint(27,41)}[trimester]

        # O'ZBEKISTON: anemiya juda keng tarqalgan (30-74%)
        if age < 19:
            anemia_level = np.random.choice([0,1,2,3], p=[0.26,0.35,0.28,0.11])  # o'smir: 74% anemik
        else:
            anemia_level = np.random.choice([0,1,2,3], p=[0.40,0.35,0.20,0.05])  # katta: 60% anemik

        # O'ZBEKISTON: temir qabul qilish past
        iron_supplement = np.random.choice([0,1], p=[0.65,0.35])

        # O'ZBEKISTON: ko'p homiladorlik qishloqlarda keng
        rural = np.random.choice([0,1], p=[0.35,0.65])  # 65% qishloqda
        if rural == 1:
            parity = np.random.choice([0,1,2], p=[0.25,0.35,0.40])  # qishloqda ko'p
        else:
            parity = np.random.choice([0,1,2], p=[0.45,0.40,0.15])  # shaharda kam

        # O'ZBEKISTON: shifokorga borish — qishloqda kamroq
        if rural == 1:
            prenatal_visits = np.random.choice([0,1,2], p=[0.25,0.45,0.30])
        else:
            prenatal_visits = np.random.choice([0,1,2], p=[0.05,0.30,0.65])

        # O'ZBEKISTON: oziqlanish muammolari
        nutrition_poor = np.random.choice([0,1], p=[0.55,0.45])

        # O'ZBEKISTON: obstetrik qon ketish tarixi
        if parity >= 2:
            pph_history = np.random.choice([0,1], p=[0.82,0.18])  # ko'p homilada yuqori
        else:
            pph_history = np.random.choice([0,1], p=[0.93,0.07])

        # O'ZBEKISTON: STI — o'smirlarda yuqori (21.5%)
        if age < 19:
            urinary_burning = np.random.choice([0,1], p=[0.78,0.22])  # STI + UTI
        else:
            urinary_burning = np.random.choice([0,1], p=[0.85,0.15])

        # Qolgan simptomlar — O'zbekiston kontekstida
        vaginal_bleeding    = np.random.choice([0,1,2], p=[0.78,0.16,0.06])
        one_sided_pain      = np.random.choice([0,1],   p=[0.93,0.07])
        nausea_severity     = np.random.choice([0,1,2,3,4], p=[0.18,0.28,0.32,0.16,0.06])
        fever               = np.random.choice([0,1,2], p=[0.83,0.12,0.05])

        # Anemiya belgisi — bosh aylanishi anemia bilan bog'liq
        if anemia_level >= 2:
            dizziness = np.random.choice([0,1,2], p=[0.30,0.40,0.30])
        else:
            dizziness = np.random.choice([0,1,2], p=[0.70,0.20,0.10])

        prev_miscarriage    = np.random.choice([0,1,2], p=[0.70,0.18,0.12])
        headache_severity   = np.random.choice([0,1,2], p=[0.68,0.22,0.10])
        visual_disturbance  = np.random.choice([0,1],   p=[0.95,0.05])
        edema_level         = np.random.choice([0,1,2,3], p=[0.52,0.28,0.15,0.05])
        fetal_movement      = np.random.choice([0,1,2], p=[0.80,0.15,0.05])
        epigastric_pain     = np.random.choice([0,1,2], p=[0.80,0.14,0.06])
        sudden_weight_gain  = np.random.choice([0,1],   p=[0.89,0.11])
        fluid_leaking       = np.random.choice([0,1],   p=[0.97,0.03])
        fetal_movement_t3   = np.random.choice([0,1,2], p=[0.76,0.18,0.06])

        # O'ZBEKISTON: erta tug'ruq o'smirlarda ko'proq (10.3%)
        if age < 19:
            contractions = np.random.choice([0,1,2], p=[0.70,0.20,0.10])
        else:
            contractions = np.random.choice([0,1,2], p=[0.76,0.17,0.07])

        bleeding_with_pain  = np.random.choice([0,1],   p=[0.96,0.04])  # obstetrik qon ketish ko'proq
        shortness_of_breath = np.random.choice([0,1,2], p=[0.75,0.20,0.05])

        # Vital belgilar
        systolic_bp  = np.random.normal(118, 16)
        diastolic_bp = np.random.normal(76, 11)
        heart_rate   = np.random.normal(84, 13)  # anemiyada yuqoriroq

        # ============================================================
        # XAVF HISOBLASH — O'zbekiston xususiy omillari qo'shilgan
        # ============================================================
        score = 0

        # --- FAVQULODDA ---
        if vaginal_bleeding==2:    score+=10
        if one_sided_pain==1:      score+=10
        if visual_disturbance==1:  score+=10
        if fetal_movement==2:      score+=10
        if fluid_leaking==1:       score+=10
        if bleeding_with_pain==1:  score+=10
        if fetal_movement_t3==2:   score+=10

        # --- YUQORI XAVF ---
        if headache_severity==2:   score+=5
        if epigastric_pain==2:     score+=5
        if contractions==2:        score+=5
        if systolic_bp>=160:       score+=5
        if systolic_bp>=140:       score+=4
        if diastolic_bp>=90:       score+=4
        if fever==2:               score+=4
        if nausea_severity==4:     score+=4
        if shortness_of_breath==2: score+=4
        if anemia_level==3:        score+=4  # O'ZB: og'ir anemiya
        if pph_history==1 and parity>=1: score+=3  # O'ZB: qon ketish tarixi

        # --- O'RTA XAVF ---
        if vaginal_bleeding==1:    score+=3
        if nausea_severity==3:     score+=3
        if urinary_burning==1:     score+=3
        if dizziness==2:           score+=3
        if fetal_movement==1:      score+=3
        if fetal_movement_t3==1:   score+=3
        if edema_level==3:         score+=3
        if anemia_level==2:        score+=2  # O'ZB: o'rta anemiya
        if headache_severity==1:   score+=2
        if edema_level==2:         score+=2
        if sudden_weight_gain==1:  score+=2
        if prev_miscarriage==2:    score+=2
        if fever==1:               score+=2
        if contractions==1:        score+=2
        if epigastric_pain==1:     score+=2
        if shortness_of_breath==1: score+=2
        if pph_history==1:         score+=2  # O'ZB: obstetrik qon ketish

        # --- PAST XAVF OMILLARI ---
        if age<18:                 score+=2  # O'ZB: erta homiladorlik
        if age>40:                 score+=1
        if parity>=2:              score+=1  # O'ZB: ko'p homiladorlik
        if rural==1 and prenatal_visits==0: score+=2  # O'ZB: qishloqda ko'rikka bormagan
        if iron_supplement==0 and anemia_level>=1: score+=1  # O'ZB: temir qabul qilmayapti
        if nutrition_poor==1:      score+=1
        if prev_miscarriage==1:    score+=1
        if anemia_level==1:        score+=1  # O'ZB: yengil anemiya
        if prenatal_visits==0:     score+=1

        risk = 'emergency' if score>=10 else 'high' if score>=6 else 'medium' if score>=3 else 'low'

        records.append({
            'trimester': trimester, 'age': age, 'gestational_week': week,
            'vaginal_bleeding': vaginal_bleeding, 'one_sided_pain': one_sided_pain,
            'nausea_severity': nausea_severity, 'urinary_burning': urinary_burning,
            'fever': fever, 'dizziness': dizziness, 'prev_miscarriage': prev_miscarriage,
            'headache_severity': headache_severity, 'visual_disturbance': visual_disturbance,
            'edema_level': edema_level, 'fetal_movement': fetal_movement,
            'epigastric_pain': epigastric_pain, 'sudden_weight_gain': sudden_weight_gain,
            'fluid_leaking': fluid_leaking, 'fetal_movement_t3': fetal_movement_t3,
            'contractions': contractions, 'bleeding_with_pain': bleeding_with_pain,
            'shortness_of_breath': shortness_of_breath,
            'systolic_bp': round(systolic_bp,1), 'diastolic_bp': round(diastolic_bp,1),
            'heart_rate': round(heart_rate,1),
            'anemia_level': anemia_level, 'iron_supplement': iron_supplement,
            'parity': parity, 'prenatal_visits': prenatal_visits,
            'nutrition_poor': nutrition_poor, 'rural': rural, 'pph_history': pph_history,
            'source': 'uzbekistan', 'risk_level': risk
        })
    return pd.DataFrame(records)

df_uzbek = generate_uzbekistan_dataset(2000)
print('O\'zbekiston dataset:', df_uzbek.shape)
print(df_uzbek['risk_level'].value_counts())
print('\nO\'zbekiston xususiy statistika:')
print(f'  Anemiya bor: {(df_uzbek.anemia_level > 0).mean():.1%}')
print(f'  Yosh onalar (<19): {(df_uzbek.age < 19).mean():.1%}')
print(f'  Qishloq: {df_uzbek.rural.mean():.1%}')
print(f'  Ko\'p homiladorlik (3+): {(df_uzbek.parity == 2).mean():.1%}')

## 🔗 3-BLOK: Datasetlarni birlashtirish

In [ ]:
# Global + O'zbekiston
df_all = pd.concat([df_global, df_uzbek], ignore_index=True)
df_all = df_all.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print(f'Jami dataset: {df_all.shape}')
print(f'  Global: {len(df_global)} | O\'zbekiston: {len(df_uzbek)}')
print(f'\nXavf taqsimoti:\n{df_all["risk_level"].value_counts()}')

# Manba bo'yicha xavf solishtirish
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

colors = {'low':'#27ae60','medium':'#f39c12','high':'#e74c3c','emergency':'#8e44ad'}
order = ['low','medium','high','emergency']

# Global vs O'zbekiston xavf
comp = df_all.groupby(['source','risk_level']).size().unstack()[order]
comp.div(comp.sum(axis=1), axis=0).plot(kind='bar', ax=axes[0],
    color=[colors[c] for c in order], edgecolor='white')
axes[0].set_title('Global vs O\'zbekiston\n(xavf foizi)', fontsize=12, fontweight='bold')
axes[0].set_xticklabels(['Global','O\'zbekiston'], rotation=0)
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'{x:.0%}'))

# Yosh bo'yicha xavf — O'zbekiston
age_bins = pd.cut(df_uzbek['age'], bins=[14,18,25,35,50], labels=['15-18','19-25','26-35','36+'])
age_risk = df_uzbek.groupby([age_bins,'risk_level']).size().unstack(fill_value=0)[order]
age_risk.div(age_risk.sum(axis=1),axis=0).plot(kind='bar', ax=axes[1],
    color=[colors[c] for c in order], edgecolor='white')
axes[1].set_title('Yosh guruhi bo\'yicha xavf\n(O\'zbekiston)', fontsize=12, fontweight='bold')
axes[1].set_xticklabels(['15-18','19-25','26-35','36+'], rotation=0)
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'{x:.0%}'))

# Anemiya va xavf — O'zbekiston
an_risk = df_uzbek.groupby(['anemia_level','risk_level']).size().unstack(fill_value=0)[order]
an_risk.div(an_risk.sum(axis=1),axis=0).plot(kind='bar', ax=axes[2],
    color=[colors[c] for c in order], edgecolor='white')
axes[2].set_title('Anemiya darajasi va xavf\n(O\'zbekiston)', fontsize=12, fontweight='bold')
axes[2].set_xticklabels(['Yo\'q','Yengil','O\'rta','Og\'ir'], rotation=0)
axes[2].yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'{x:.0%}'))

plt.tight_layout()
plt.savefig('/content/uzbekistan_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ O\'zbekiston tahlil grafigi saqlandi')

## 📊 4-BLOK: Kaggle dataseti (mavjud bo'lsa)

In [ ]:
KAGGLE_PATH = '/content/drive/MyDrive/onamiz_data/Maternal Health Risk Data Set.csv'
FEATURE_COLS = [
    'trimester_enc', 'age', 'gestational_week',
    'vaginal_bleeding', 'one_sided_pain', 'nausea_severity',
    'urinary_burning', 'fever', 'dizziness', 'prev_miscarriage',
    'headache_severity', 'visual_disturbance', 'edema_level',
    'fetal_movement', 'epigastric_pain', 'sudden_weight_gain', 'fluid_leaking',
    'fetal_movement_t3', 'contractions', 'bleeding_with_pain', 'shortness_of_breath',
    'systolic_bp', 'diastolic_bp', 'heart_rate',
    # O'zbekiston xususiy (yangi!)
    'anemia_level', 'iron_supplement', 'parity',
    'prenatal_visits', 'nutrition_poor', 'rural', 'pph_history'
]

if os.path.exists(KAGGLE_PATH):
    kdf = pd.read_csv(KAGGLE_PATH)
    print('Kaggle dataset:', kdf.shape)
    kdf_mapped = pd.DataFrame({
        'trimester': 'T2', 'age': kdf['Age'], 'gestational_week': 20,
        'systolic_bp': kdf['SystolicBP'], 'diastolic_bp': kdf['DiastolicBP'],
        'heart_rate': kdf['HeartRate'],
        'fever': (kdf['BodyTemp'] > 99.5).astype(int),
        **{c: 0 for c in FEATURE_COLS if c not in
           ['trimester_enc','age','gestational_week','systolic_bp','diastolic_bp','heart_rate','fever']},
        'source': 'kaggle',
        'risk_level': kdf['RiskLevel'].map({'low risk':'low','mid risk':'medium','high risk':'high'})
    })
    df_all = pd.concat([df_all, kdf_mapped], ignore_index=True)
    print(f'Kaggle qo\'shildi. Jami: {df_all.shape}')
else:
    print('Kaggle topilmadi — faqat sintetik va O\'zbekiston dataseti ishlatiladi.')
    print(f'Jami dataset: {df_all.shape}')

## 🏋️ 5-BLOK: Model o'qitish

In [ ]:
le_trim = LabelEncoder()
df_all['trimester_enc'] = le_trim.fit_transform(df_all['trimester'])

le_y = LabelEncoder()
df_all['risk_enc'] = le_y.fit_transform(df_all['risk_level'])

print('Sinflar:', dict(enumerate(le_y.classes_)))
print(f'Features: {len(FEATURE_COLS)} ta')

X = df_all[FEATURE_COLS].fillna(0)
y = df_all['risk_enc']

scaler = StandardScaler()
X_sc = scaler.fit_transform(X)

smote = SMOTE(random_state=RANDOM_STATE)
X_bal, y_bal = smote.fit_resample(X_sc, y)

print('\nSMOTE dan keyin:')
print(pd.Series(le_y.inverse_transform(y_bal)).value_counts())

X_tr, X_te, y_tr, y_te = train_test_split(
    X_bal, y_bal, test_size=0.2, stratify=y_bal, random_state=RANDOM_STATE
)
print(f'\nTrain: {X_tr.shape[0]} | Test: {X_te.shape[0]}')

In [ ]:
candidates = {
    'RandomForest': RandomForestClassifier(n_estimators=300, max_depth=10, random_state=RANDOM_STATE, n_jobs=-1),
    'XGBoost':      XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, eval_metric='mlogloss', random_state=RANDOM_STATE, n_jobs=-1),
    'LightGBM':     LGBMClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1),
    'GradientBoosting': GradientBoostingClassifier(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=RANDOM_STATE)
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
results = {}
print('=' * 60)
for name, model in candidates.items():
    sc = cross_val_score(model, X_tr, y_tr, cv=cv, scoring='f1_macro', n_jobs=-1)
    results[name] = {'mean': sc.mean(), 'std': sc.std()}
    print(f'  {name:22s}  F1={sc.mean():.4f} (±{sc.std():.4f})')
best_name = max(results, key=lambda k: results[k]['mean'])
print(f'\n  🏆 Eng yaxshi: {best_name}')

best_model = candidates[best_name]
best_model.fit(X_tr, y_tr)
y_pred = best_model.predict(X_te)
acc = accuracy_score(y_te, y_pred)
f1  = f1_score(y_te, y_pred, average='macro')
print(f'\nTest Accuracy: {acc:.4f}')
print(f'Test F1-macro: {f1:.4f}')
print(classification_report(y_te, y_pred, target_names=le_y.classes_))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cm = confusion_matrix(y_te, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le_y.classes_, yticklabels=le_y.classes_, ax=axes[0])
axes[0].set_title(f'Onamiz v3 — {best_name}\nAccuracy={acc:.3f} | F1={f1:.3f}', fontweight='bold')
axes[0].set_xlabel('Bashorat'); axes[0].set_ylabel('Haqiqiy')

# Feature importance
if hasattr(best_model, 'feature_importances_'):
    imp = pd.Series(best_model.feature_importances_, index=FEATURE_COLS).nlargest(15)
    imp.plot(kind='barh', ax=axes[1], color='steelblue')
    axes[1].set_title('Top 15 muhim feature', fontweight='bold')
    axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('/content/model_v3_results.png', dpi=150)
plt.show()

## 💾 6-BLOK: Saqlash va test

In [ ]:
os.makedirs('/content/models', exist_ok=True)
artifact = {
    'model': best_model, 'scaler': scaler,
    'label_encoder': le_y, 'trimester_encoder': le_trim,
    'feature_names': FEATURE_COLS, 'class_labels': list(le_y.classes_),
    'best_model_name': best_name,
    'test_accuracy': float(acc), 'test_f1_macro': float(f1),
    'dataset_info': {'global': 3000, 'uzbekistan': 2000, 'total': len(df_all)},
    'uzbekistan_features': ['anemia_level','iron_supplement','parity','prenatal_visits','nutrition_poor','rural','pph_history'],
    'sources': ['WHO ANC 2016','ACOG 2023','FIGO 2022','EPDS','Andijan Maternal Mortality Study 2025','WHO Uzbekistan Statistics','Demografiya.uz 2021']
}
joblib.dump(artifact, '/content/models/onamiz_v3.joblib')

for f in ['onamiz_v3.joblib']:
    shutil.copy(f'/content/models/{f}', f'/content/drive/MyDrive/onamiz_models/{f}')
for f in ['uzbekistan_analysis.png', 'model_v3_results.png']:
    if os.path.exists(f'/content/{f}'):
        shutil.copy(f'/content/{f}', f'/content/drive/MyDrive/onamiz_models/{f}')

print(f'✅ Model saqlandi: onamiz_v3.joblib')
print(f'   Aniqlik: {acc:.1%} | F1: {f1:.4f}')
print(f'   Features: {len(FEATURE_COLS)} ta (7 ta O\'zbekiston xususiy)')
print(f'   Dataset: {len(df_all)} ta namuna')

In [ ]:
def predict(inp):
    msgs = {
        'low':       ('🟢','yashil',     'Ko\'rsatkichlar normal. Rejali ko\'rikka boring.'),
        'medium':    ('🟡','sariq',      'Ba\'zi belgilar bor. Bu hafta shifokorni ko\'ring.'),
        'high':      ('🔴','qizil',      'BUGUN shifokorga boring.'),
        'emergency': ('🚨','favqulodda', 'TEZKOR: Hoziroq tez yordam!')
    }
    row = {c: inp.get(c,0) for c in FEATURE_COLS}
    X_in = pd.DataFrame([row])
    probs = best_model.predict_proba(scaler.transform(X_in))[0]
    cls = le_y.classes_[np.argmax(probs)]
    e, r, m = msgs[cls]
    return {'risk': r, 'emoji': e, 'message': m,
            'probs': {c: round(float(p),3) for c,p in zip(le_y.classes_, probs)}}

print('=' * 65)
print('  O\'ZBEKISTON STSENARIYLAR TESTI')
print('=' * 65)

tests = [
    ('✅ Sog\'lom (28 yosh, shahar, temir qabul qilyapti)',
     {'trimester_enc':1,'age':28,'gestational_week':18,'systolic_bp':115,'diastolic_bp':75,'heart_rate':80,
      'anemia_level':0,'iron_supplement':1,'parity':0,'prenatal_visits':2,'nutrition_poor':0,'rural':0,'pph_history':0}),

    ('⚠️ 17 yoshli, qishloqda, anemiyasi bor, shifokorga bormagan',
     {'trimester_enc':0,'age':17,'gestational_week':10,'systolic_bp':108,'diastolic_bp':68,'heart_rate':96,
      'anemia_level':2,'iron_supplement':0,'parity':0,'prenatal_visits':0,'nutrition_poor':1,'rural':1,
      'dizziness':2,'nausea_severity':3,'urinary_burning':1,'pph_history':0}),

    ('🔴 Ko\'p homiladorlik, qon ketish tarixi, anemiya',
     {'trimester_enc':2,'age':34,'gestational_week':32,'systolic_bp':138,'diastolic_bp':88,'heart_rate':92,
      'anemia_level':2,'iron_supplement':0,'parity':2,'prenatal_visits':1,'nutrition_poor':1,'rural':1,
      'pph_history':1,'headache_severity':1,'edema_level':2,'fetal_movement_t3':1}),

    ('🚨 Ko\'z oldida uchish + kuchli bosh og\'riq (eklampsiya)',
     {'trimester_enc':2,'age':35,'gestational_week':34,'systolic_bp':165,'diastolic_bp':112,'heart_rate':102,
      'visual_disturbance':1,'headache_severity':2,'edema_level':3,'epigastric_pain':2,
      'anemia_level':1,'iron_supplement':0,'parity':1,'prenatal_visits':1,'nutrition_poor':0,'rural':0,'pph_history':0}),

    ('🚨 Og\'ir anemiya + qon ketish (obstetrik favqulodda)',
     {'trimester_enc':2,'age':30,'gestational_week':36,'systolic_bp':120,'diastolic_bp':78,'heart_rate':108,
      'anemia_level':3,'iron_supplement':0,'parity':2,'prenatal_visits':0,'nutrition_poor':1,'rural':1,
      'bleeding_with_pain':1,'pph_history':1,'dizziness':2,'shortness_of_breath':1})
]

for name, inp in tests:
    r = predict(inp)
    print(f'\n📋 {name}')
    print(f'   {r["emoji"]} {r["risk"].upper()}: {r["message"]}')
    print(f'   📊 {r["probs"]}')

## ✅ Xulosa — v3

| | v2 (faqat global) | v3 (+ O'zbekiston) |
|---|---|---|
| Dataset hajmi | 3,000 | **5,000+** |
| Features | 24 ta | **31 ta** |
| O'zbekiston omillari | ❌ | ✅ Anemiya, temir, parity, qishloq |
| Yosh onalar (15-18) | Oz | **Ko'p (real statistika)** |
| Manba | WHO/ACOG | + **SSV, Demografiya.uz, Andijan tadqiqoti** |

### Yangi 7 ta O'zbekiston-xususiy feature:
1. `anemia_level` — Kamqonlik darajasi (0-3)
2. `iron_supplement` — Temir qabul qilyaptimi
3. `parity` — Nechta homiladorlik
4. `prenatal_visits` — Shifokorga necha marta borgan
5. `nutrition_poor` — Oziqlanish holati
6. `rural` — Qishloqda yashaydimi
7. `pph_history` — Avval obstetrik qon ketganmi